In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import utils

plt.style.use('default')

In [ ]:
df = utils.union_read_csv('data/Gson_generation-validity.csv', 'data/Lang_generation-validity.csv',
                          'data/JacksonCore_generation-validity.csv')
gen_validity = df.sum()

In [ ]:
total_generations = gen_validity['TOTAL_GENERATIONS']
parse_fails = gen_validity['PARSE_FAILURES']
compile_fails = gen_validity['COMPILE_FAILURES']
test_fails = gen_validity['TEST_FAILURES']
other_fails = gen_validity['OTHER_FAILURES']
valid_fails = gen_validity['VALID_GENERATIONS']

failure_color_dark = '#b00'
success_color_dark = '#080'
failure_color_light = '#d33'
success_color_light = '#3d3'

content = f''':Generations {success_color_dark}

Generations [{parse_fails}] Parse Fail {failure_color_light}
Generations [{total_generations - parse_fails}] Parse Success {success_color_light}

:Parse Fail {failure_color_dark}
:Parse Success {success_color_dark}

Parse Success [{compile_fails}] Compile Fail {failure_color_light}
Parse Success [*] Compile Success {success_color_light}

:Compile Fail {failure_color_dark}
:Compile Success {success_color_dark}

Compile Success [{test_fails}] Test Fail {failure_color_light}
Compile Success [*] Test Success {success_color_light}

:Test Fail {failure_color_dark}
:Test Success {success_color_dark}

Parse Fail [*] Invalid {failure_color_light}
Compile Fail [*] Invalid {failure_color_light}
Test Fail [*] Invalid {failure_color_light}

Test Success [*] Valid {success_color_light}

:Invalid {failure_color_dark}
:Valid {success_color_dark}
'''

with open('out/generation-validity.sankey.txt', 'w') as f:
    f.write(content)

In [ ]:
df = utils.union_read_csv('data/Gson_iterations.csv', 'data/Lang_iterations.csv', 'data/JacksonCore_iterations.csv')
df = df[df['RUN_COMPLETED'] == True]

# Do not count iteration 0 twice per run
df['TYPE'] = df.apply(lambda row: 'INITIAL' if row['ITERATION'] == 0 else row['FEEDBACK_TYPE'], axis=1)
df = df[['RUN_UUID', 'CLASS_NAME', 'TYPE', 'ITERATION', 'FIX_REPROMPT_COUNT']].drop_duplicates(['RUN_UUID', 'CLASS_NAME', 'TYPE', 'ITERATION'])

values = df['FIX_REPROMPT_COUNT']

In [ ]:
bins = {f'{n}': values.where(values == n).count() for n in range(6)}
bins['$> 5$'] = values.where(values >= 6).count()

total_generations = sum(v for v in bins.values())
print(f'{total_generations} Generations')

figsize = utils.A5_LANDSCAPE_INCHES
figsize = (figsize[0], figsize[1] / 3 * 2)
fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)
fig.suptitle('Verteilung der Reprompt-Anzahl pro Iteration')

bars = axes.bar(bins.keys(), bins.values())

FLIP = 1500
for bar in bars:
    height = bar.get_height()
    x = bar.get_x()
    width = bar.get_width()
    axes.annotate(
        f'{height:,.0f}'.replace(',', ' '),
        xy=(x + width / 2.0, height),
        xytext=(0, 1),
        textcoords='offset points',
        ha='center',
        va='bottom',
    )
    txt = axes.text(
        x + width / 2.0,
        height + (1100 if height < FLIP else -250),
        f"{height * 100.0 / total_generations:.0f} %",
        ha='center',
        va='bottom' if height < FLIP else 'top',
        fontsize=8,
    )
    if height >= FLIP:
        txt.set_bbox(dict(facecolor='white', alpha=0.5, edgecolor='none', pad=3))

axes.set_xlabel('Anzahl Reprompts')
axes.set_ylabel('Häufigkeit (Anzahl Iterationen)')
axes.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, pos: f'{val:,.0f}'.replace(',', ' ')))
axes.margins(y=.1)
axes.grid(linestyle='--', linewidth=.5, alpha=0.5, axis='y')
plt.show()
fig.savefig(f'out/fix-reprompt-counts.svg', bbox_inches='tight')